1. Scopo e logica della procedura

Nel primo passaggio si costruisce un dataset “master” dei box office domestic, aggregando i file annuali in un’unica tabella coerente. La standardizzazione riguarda: (i) uniformità dei nomi colonna, (ii) normalizzazione di variabili monetarie e numeriche, (iii) estrazione dell’anno di riferimento (“box_office_year”) dai file annuali, così da rendere il dataset interrogabile longitudinalmente.

In [1]:
import os
import re
import glob
import pandas as pd
from datetime import datetime

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


2. Individuazione dei file annuali

I file annuali vengono individuati tramite pattern sul percorso. La procedura assume una convenzione stabile dei nomi file (es. box_office_domestic_top100_1980.csv) e permette di estendere la pipeline senza modifiche qualora vengano aggiunti ulteriori anni.

In [2]:
DATA_DIR = "data/box_office_domestic"
pattern = os.path.join(DATA_DIR, "box_office_domestic_top100_*.csv")
files = sorted(glob.glob(pattern))

print("Numero file trovati:", len(files))
print("Esempio primi 5:", files[:5])
print("Esempio ultimi 5:", files[-5:])


Numero file trovati: 41
Esempio primi 5: ['data/box_office_domestic/box_office_domestic_top100_1980.csv', 'data/box_office_domestic/box_office_domestic_top100_1981.csv', 'data/box_office_domestic/box_office_domestic_top100_1982.csv', 'data/box_office_domestic/box_office_domestic_top100_1983.csv', 'data/box_office_domestic/box_office_domestic_top100_1984.csv']
Esempio ultimi 5: ['data/box_office_domestic/box_office_domestic_top100_2016.csv', 'data/box_office_domestic/box_office_domestic_top100_2017.csv', 'data/box_office_domestic/box_office_domestic_top100_2018.csv', 'data/box_office_domestic/box_office_domestic_top100_2019.csv', 'data/box_office_domestic/box_office_domestic_top100_2020.csv']


3. Funzioni di normalizzazione (valute e interi)

Per garantire comparabilità, le variabili monetarie vengono convertite in interi (USD), rimuovendo simboli e separatori. Analogamente, i conteggi di biglietti (“tickets sold”) vengono convertiti in interi. Questa normalizzazione riduce ambiguità e facilita aggregazioni, regressioni e analisi longitudinali.

In [3]:
def parse_money_to_int(x):
    """Converte stringhe tipo '$181,353,855' -> 181353855. Restituisce NA se vuoto/non valido."""
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return pd.NA
    s = s.replace("$", "").replace(",", "").strip()
    if s == "":
        return pd.NA
    try:
        return int(s)
    except:
        return pd.NA

def parse_int(x):
    """Converte stringhe con separatori (es. '67,417,790') in int."""
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return pd.NA
    s = s.replace(",", "").strip()
    try:
        return int(s)
    except:
        return pd.NA


4. Lettura, standardizzazione e concatenazione

Ogni file annuale viene letto, normalizzato e arricchito con una variabile box_office_year, derivata dal nome del file. Si produce quindi una concatenazione verticale che preserva l’informazione temporale e assicura uno schema uniforme.

In [4]:
def extract_year_from_filename(fp):
    m = re.search(r"top100_(\d{4})\.csv$", fp)
    if not m:
        return None
    return int(m.group(1))

required_cols_candidates = {
    "rank": ["rank"],
    "title": ["title"],
    "release_date": ["release_date", "release date"],
    "distributor": ["distributor"],
    "genre": ["genre"],
    "domestic_gross_raw": ["domestic_gross_raw", "domestic_gross", "domestic gross", "domestic_gross_"],
    "tickets_sold_raw": ["tickets_sold_raw", "tickets_sold", "tickets sold", "tickets_sold_"]
}

def standardize_columns(df):
    # normalizza nomi colonna: lower + underscore
    df = df.copy()
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    return df

def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

rows = []
for fp in files:
    year = extract_year_from_filename(fp)
    if year is None:
        print("ATTENZIONE: anno non estratto da", fp)
        continue

    df = pd.read_csv(fp)
    df = standardize_columns(df)

    # mappatura colonne effettive -> standard
    colmap = {}
    for std, cands in required_cols_candidates.items():
        found = pick_col(df, cands)
        if found is not None:
            colmap[found] = std

    df = df.rename(columns=colmap)

    # colonne minime attese (se mancanti, le creiamo come NA)
    for std in ["rank", "title", "release_date", "distributor", "domestic_gross_raw", "tickets_sold_raw"]:
        if std not in df.columns:
            df[std] = pd.NA

    df["box_office_year"] = year

    # parsing numerici
    df["rank"] = df["rank"].apply(parse_int)
    df["domestic_box_office_gross"] = df["domestic_gross_raw"].apply(parse_money_to_int)
    df["tickets_sold"] = df["tickets_sold_raw"].apply(parse_int)

    # release_year: estrazione prudente dall'ultima parte della stringa data o dal box_office_year come fallback
    # (non forziamo parsing completo della data, perché i formati possono variare)
    df["release_year"] = df["release_date"].astype(str).str.extract(r"(\d{4})").astype("Int64")
    df.loc[df["release_year"].isna(), "release_year"] = year  # fallback controllato

    # selezione colonne master (incluso raw per audit)
    keep = [
        "box_office_year", "rank", "title", "release_date", "release_year",
        "distributor", "genre",
        "domestic_gross_raw", "domestic_box_office_gross",
        "tickets_sold_raw", "tickets_sold"
    ]
    df = df[keep]
    rows.append(df)

boxoffice_master = pd.concat(rows, ignore_index=True)
print("Shape master:", boxoffice_master.shape)
boxoffice_master.head(10)


Shape master: (4100, 11)


,box_office_year,rank,title,release_date,release_year,distributor,genre,domestic_gross_raw,domestic_box_office_gross,tickets_sold_raw,tickets_sold
0,1980,1,Star Wars Ep. V: The Empire Strikes Back,"May 21, 1980",1980,20th Century Fox,Adventure,"$181,353,855",181353855,"67,417,790",67417790
1,1980,2,Stir Crazy,"Dec 12, 1980",1980,Columbia,Black Comedy,"$101,300,000",101300000,"37,657,992",37657992
2,1980,3,Kramer vs. Kramer,"Dec 19, 1979",1979,NaN,Drama,"$98,982,763",98982763,"36,796,566",36796566
3,1980,4,Airplane!,"Jul 4, 1980",1980,Paramount Pictures,Comedy,"$83,453,539",83453539,"31,023,620",31023620
4,1980,5,Any Which Way You Can,"Dec 17, 1980",1980,Warner Bros.,Comedy,"$70,687,344",70687344,"26,277,823",26277823
5,1980,6,Private Benjamin,"Oct 10, 1980",1980,Warner Bros.,Comedy,"$69,847,348",69847348,"25,965,556",25965556
6,1980,7,Coal Miner's Daughter,"Mar 7, 1980",1980,Universal,Drama,"$67,182,787",67182787,"24,975,013",24975013
7,1980,8,Smokey and the Bandit II,"Aug 15, 1980",1980,Universal,Comedy,"$66,132,626",66132626,"24,584,619",24584619
8,1980,9,The Blues Brothers,"Jun 20, 1980",1980,Universal,Comedy,"$57,229,890",57229890,"21,275,052",21275052
9,1980,10,Ordinary People,"Sep 19, 1980",1980,Paramount Pictures,Drama,"$52,302,978",52302978,"19,443,486",19443486


5. Controlli di qualità (QA)

Prima di salvare il master, si eseguono controlli essenziali: presenza di null, duplicati (per anno+rank), distribuzione degli anni e verifica dei campi numerici. Questa fase serve a intercettare inconsistenze prima delle operazioni successive (lookup OMDb e merge metadati).

In [5]:
# anni coperti
print("Anni min/max:", boxoffice_master["box_office_year"].min(), boxoffice_master["box_office_year"].max())
print("N anni distinti:", boxoffice_master["box_office_year"].nunique())

# duplicati su chiave logica: (box_office_year, rank)
dups = boxoffice_master.duplicated(subset=["box_office_year", "rank"], keep=False).sum()
print("Duplicati su (box_office_year, rank):", dups)

# null principali
print("\nNull principali:")
print(boxoffice_master[["title","distributor","domestic_box_office_gross","tickets_sold"]].isna().mean().sort_values(ascending=False))

# sanity check numerici
print("\nEsempio righe con gross mancante:")
display(boxoffice_master[boxoffice_master["domestic_box_office_gross"].isna()].head(5))


Anni min/max: 1980 2020
N anni distinti: 41
Duplicati su (box_office_year, rank): 0

Null principali:
distributor                  0.013415
tickets_sold                 0.000732
title                        0.000000
domestic_box_office_gross    0.000000
dtype: float64

Esempio righe con gross mancante:


,box_office_year,rank,title,release_date,release_year,distributor,genre,domestic_gross_raw,domestic_box_office_gross,tickets_sold_raw,tickets_sold


6. Esportazione del master

Il dataset “master” viene salvato in formato CSV in una posizione stabile del progetto, così da diventare input diretto per le fasi successive (match IMDb via OMDb e costruzione metadata finale).

In [6]:
OUT_MASTER = "data/boxoffice_master_domestic.csv"
os.makedirs("data", exist_ok=True)
boxoffice_master.to_csv(OUT_MASTER, index=False)
print("Salvato:", OUT_MASTER)


Salvato: data/boxoffice_master_domestic.csv


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=e08cfdf8-9b6e-44e8-b36c-3dd235d85ba1' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>